### TODO:
1. Load multiple dataset for different buildings.
2. Get the climate info for the state.
3. preprocess the data.
4. Create a linear regression model.
5. Prepare some tests for the model.

In [23]:
import polars as pl
import pyarrow
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path
folder= Path().resolve().parent /'input'


### 1. Load multiple dataset for different buildings.

In [3]:
# read the original data for three individual buildings
data=pl.scan_parquet(folder / 'load_profile_buildingID_*')

In [4]:
# extract the buildings Ids
bldgs=data.select(pl.col('bldg_id').unique()).collect().to_series().to_list()

### 2. Get the climate info for the state.

In [5]:
# extract the climate zone details based on the builings and cast the building id type to match the one in the original data
MetaData=(
    pl.scan_parquet(folder / 'MetaData.parquet')
            .filter(
            pl.col('bldg_id').is_in(bldgs)).cast({"bldg_id": pl.UInt32}).select(
            pl.col('bldg_id', 'in.ashrae_iecc_climate_zone_2004_sub_cz_split'))
)
# MetaData.row(0, named=True) 
MetaData.collect()

bldg_id,in.ashrae_iecc_climate_zone_2004_sub_cz_split
u32,str
10,"""1A - FL"""
100016,"""2A - FL, GA, AL, MS"""
100024,"""1A - FL"""


In [6]:
%%time
# merge the climate zone changes into the original data
df=data.join(MetaData, on='bldg_id')

CPU times: user 279 μs, sys: 53 μs, total: 332 μs
Wall time: 365 μs


There are only time climate zones for the data

----------------------------------------------

### Prepare cross-validation model


In [ ]:
df.collect_schema()

In [8]:
# defining the independant and dependant variables with encoding categorical variables and converting the datetime to a timestamp
x= df.select(pl.col("timestamp").dt.timestamp(), pl.col('out.electricity.total.energy_consumption..kwh').alias('Total Usage'),
             pl.col('in.ashrae_iecc_climate_zone_2004_sub_cz_split').cast(pl.Categorical).cast(pl.UInt32).alias('Climate Zone')).collect()
y=df.select(pl.col("out.electricity.cooling.energy_consumption..kwh").alias('Cooling Consumption'),
           pl.col("out.electricity.heating.energy_consumption..kwh").alias('Heating Consumption'),
           pl.col("out.electricity.freezer.energy_consumption..kwh").alias('Freezing Consumption'),
           pl.col("out.electricity.refrigerator.energy_consumption..kwh").alias('Refrigerator Consumption'),
           pl.col("out.electricity.dishwasher.energy_consumption..kwh").alias('Dishwasher Consumption')).collect()

In [9]:
x.head(6)

timestamp,Total Usage,Climate Zone
i64,f32,u32
1514765700000000,0.1662,0
1514766600000000,0.15641,0
1514767500000000,0.14339,0
1514768400000000,0.14249,0
1514769300000000,0.14133,0
1514770200000000,0.53151,0


In [10]:
y.head(6)

Cooling Consumption,Heating Consumption,Freezing Consumption,Refrigerator Consumption,Dishwasher Consumption
f32,f32,f32,f32,f32
0.0,0.05603,0.0,0.01194,0.0
0.0,0.04812,0.0,0.01194,0.0
0.0,0.04185,0.0,0.01194,0.0
0.0,0.0338,0.0,0.01194,0.0
0.0,0.04049,0.0,0.01092,0.0
0.0,0.04627,0.0,0.01092,0.0


In [ ]:
# confirming types
x.schema

In [38]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
def model(mod, x_train, x_test, y_train, y_test):
    mod.fit(x_train, y_train)
    predicted=mod.predict(x_test)
    # return mod.score(x_test, y_test)
    # return r2_score(y_test, predicted)
    # return mean_absolute_error(y_test, predicted)
    # return mean_squared_error(y_test, predicted)
    return root_mean_squared_error(y_test, predicted)

In [39]:
%%time
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xg
skf= StratifiedKFold(n_splits=6)
kf=KFold(n_splits=10)
arr=[]
lf=pd.DataFrame()
for i, (train,test) in enumerate(kf.split(x,y)):
    x_train, x_test, y_train, y_test=x[train], x[test], y[train], y[test]
    LR=model(LinearRegression(), x_train, x_test, y_train, y_test)
    RF=model(RandomForestRegressor(n_estimators=3, max_depth=2, n_jobs=-1), x_train, x_test, y_train, y_test)
    XG=model(xg.XGBRegressor(), x_train, x_test, y_train, y_test)
    frame=pd.DataFrame({
    "Split Number": [i],
    "Linear Regression": [LR],
    "Random Forest": [RF],
    "XG Boost": [XG]
    })
    arr.append(frame)
lf=pd.concat(arr)

CPU times: user 5min 14s, sys: 434 ms, total: 5min 14s
Wall time: 22.1 s


In [40]:
lf.mean()

Split Number         4.500000
Linear Regression    0.040428
Random Forest        0.036776
XG Boost             0.034962
dtype: float64

In [17]:
## cross val score
LR=LinearRegression()
RF=RandomForestRegressor(n_estimators=3, max_depth=2, n_jobs=-1)
XG=xg.XGBRegressor()

In [24]:
np.mean(cross_val_score(LR, x, y, cv=10))

np.float64(-0.4986558336942439)

In [25]:
np.mean(cross_val_score(RF, x, y, cv=10))

np.float64(-0.26515005816013615)

In [26]:
np.mean(cross_val_score(XG, x, y, cv=10))

np.float64(-0.8510457657277584)

In [14]:
lf.mean()

Split Number         4.500000
Linear Regression   -0.498656
Random Forest       -0.271530
XG Boost            -0.851046
dtype: float64